# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² Kilifi County Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, accessible from a public URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\nIdentifier: {metadata.identifier}\nLicense: {metadata.license}")

## 2. Data Overview
List available record sets, and for each record set show its `@id`, name, and available fields/columns with their `@id`s.

In [ ]:
# Inspect the record sets
record_sets = list(dataset.metadata.record_sets)

print(f"Found {len(record_sets)} record set(s):\n")
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet: {rs.id}")
    print(f"    Name: {rs.name}")
    print(f"    Description: {getattr(rs, 'description', None)}")
    print("    Fields/columns:")
    for field in getattr(rs, 'fields', []):
        print(f"        - {field.id}")
    for col in getattr(rs, 'columns', []):
        print(f"        - {col.id}")
    print("")
if not record_sets:
    print("No record sets found directly in the Croissant metadata. Trying to peek at available records...")
    # Try to list records (may discover the record_set id)
    try:
        sample = dataset.records()
        sample_list = list(sample)
        print(f"Sample records: {sample_list[:1]}")
    except Exception as e:
        print(f"Could not enumerate records: {e}")

### Manual listing for convenience
Based on inspection, here is a possible main record set:

- `cr:RecordSet/0` (the primary survey record set; please confirm name using the output above)

*Replace all IDs below with the actual `@id` strings from metadata overview above if they differ,*
but for demonstration we'll proceed assuming one main survey record set.

## 3. Data Extraction
Load data from the main record set and inspect the first few records. Reference the record set and fields via their `@id`s.

In [ ]:
# Define the record set ID (replace with the appropriate ID found above)
main_record_set_id = None
if record_sets:
    main_record_set_id = record_sets[0].id
else:
    # Fallback value (example): 'cr:RecordSet/0'
    main_record_set_id = 'cr:RecordSet/0'

print(f"Loading records from record set: {main_record_set_id}")

main_records = list(dataset.records(record_set=main_record_set_id))
main_df = pd.DataFrame(main_records)
print(f"Columns in main DataFrame:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter the records, normalize a numeric field, and group by a demographic attribute.

We select fields based on their `@id` (column names), e.g. GAD-7 total score (`gad7_score` or similar), PHQ-9 (`phq9_score`) or PSQ (`psq_score`), and a demographic group like `gender`.

First, we print all column names to locate IDs that look like total test scores or demographic attributes.

In [ ]:
# Display columns for manual selection
print(main_df.columns.tolist())
# Example: use 'gad7_total' for numeric_field, 'gender' for group_field
# Be sure to replace with the exact @id if columns differ.

numeric_field = None
for c in main_df.columns:
    if 'gad' in c.lower() and ('score' in c.lower() or 'total' in c.lower()):
        numeric_field = c; break
if not numeric_field:
    for c in main_df.columns:
        if 'phq' in c.lower() and ('score' in c.lower() or 'total' in c.lower()):
            numeric_field = c; break
if not numeric_field:
    for c in main_df.columns:
        if 'psq' in c.lower() and ('score' in c.lower() or 'total' in c.lower()):
            numeric_field = c; break
if not numeric_field:
    numeric_field = main_df.select_dtypes('number').columns[0]
print(f"Using numeric field for EDA: {numeric_field}")

group_field = None
for c in main_df.columns:
    if c.lower() in ('gender','sex','participant_gender'):
        group_field = c; break
if not group_field:
    for c in main_df.columns:
        if 'age' in c.lower():
            group_field = c; break
if not group_field:
    group_field = main_df.columns[0]
print(f"Using group field: {group_field}")

threshold = main_df[numeric_field].mean()

filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} samples")
print(filtered_df[[numeric_field, group_field]].head())

# Normalize
filtered_df[numeric_field+'_normalized'] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() + 1e-8)
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    print(grouped_df[[numeric_field, numeric_field+'_normalized']].head())

## 5. Visualization
Plot the distribution of the main numeric field and compare by demographic group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=20, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Group by demographic group
if group_field in main_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=main_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library.
- Inspected schema-level record sets, their fields, and all metadata by `@id`.
- Loaded the main survey table via its `@id` and explored available fields.
- Performed exploratory data analysis, filtering records, normalizing a numeric field, and grouping by a demographic column (referenced by `@id`).
- Visualized key distributions and demographic relationships.

This analysis serves as a reproducible foundation for further, more detailed analytical workflows using FAIR, AI-ready survey datasets in the Croissant schema.
